In [10]:
import os
import random
import string
import dedupe
import dedupe.variables
import json
import pandas as pd
import datetime

# ==========================================
# 1. Complex Data Generator 🌪️
# ==========================================
def random_date(start_year=1960, end_year=2000):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s, intensity=1):
    """Introduces random typos: swaps, deletions, or replacements."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    for _ in range(intensity):
        idx = random.randint(0, len(s_list) - 2)
        # Randomly choose corruption type: 0=Swap, 1=Delete, 2=Replace
        ctype = random.choice([0, 0, 1]) 
        if ctype == 0: # Swap
            s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
        elif ctype == 1: # Delete
            del s_list[idx]
    return "".join(s_list)

def corrupt_phone(phone):
    """Changes formatting: (123) 456-7890 -> 123.456.7890"""
    digits = "".join(filter(str.isdigit, phone))
    style = random.choice(['dot', 'plain', 'paren'])
    if style == 'dot': return f"{digits[:3]}.{digits[3:6]}.{digits[6:]}"
    if style == 'plain': return digits
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

def generate_complex_dataset(num_base_records=70, duplicate_ratio=0.5):
    
    # --- Data Pools ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct"]
    
    data_d = {}
    ground_truth = {}
    counter = 0

    # --- 1. Base Records ---
    for _ in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        # Phone: Standard XXX-XXX-XXXX
        phone = f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}"
        dob = random_date()
        
        record = {
            'name': f"{fn} {ln}",
            'address': f"{random.randint(100, 999)} {random.choice(streets)}",
            'city': random.choice(["New York", "Chicago", "Los Angeles", "Houston"]),
            'phone': phone,
            'email': f"{fn[0].lower()}{ln.lower()}@{random.choice(domains)}",
            'dob': dob.strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]

    # --- 2. Generate Noisy Duplicates ---
    num_dupes = int(num_base_records * duplicate_ratio)
    ids_to_dupe = random.sample(list(data_d.keys()), num_dupes)

    for original_id in ids_to_dupe:
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # A. Complex Name Noise (Initials or Typos)
        if random.random() > 0.5:
            parts = new_rec['name'].split()
            new_rec['name'] = f"{parts[0][0]}. {parts[1]}" # J. Smith
        else:
            new_rec['name'] = corrupt_string(new_rec['name']) # Jhon Smith

        # B. Phone Formatting Noise
        new_rec['phone'] = corrupt_phone(new_rec['phone'])

        # C. Email Typos
        if random.random() > 0.7:
            new_rec['email'] = corrupt_string(new_rec['email']) # jsmith@gmil.com
        
        # D. DOB Format Swap (US vs UK style)
        if random.random() > 0.8:
            y, m, d = new_rec['dob'].split('-')
            new_rec['dob'] = f"{d}/{m}/{y}"

        # E. Missing Data (Nulls)
        if random.random() > 0.8: new_rec['zip'] = None
        if random.random() > 0.9: new_rec['email'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- 3. Generate Training Pairs ---
    match_pairs, distinct_pairs = [], []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(20):
        if not multi_grps: break
        grp = random.choice(multi_grps)
        a, b = random.sample(grp, 2)
        match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(20):
        a, b = random.sample(all_ids, 2)
        # Ensure not same group
        is_same = False
        for g in ground_truth.values():
            if a in g and b in g: is_same = True
        if not is_same:
            distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 2. Main Logic
# ==========================================
SETTINGS_FILE = 'complex_dedupe.settings'
TRAINING_FILE = 'complex_dedupe.json'

print("🌪️ Generating Complex Dataset (Typos, Formats, Nulls)...")
data_d, training_data = generate_complex_dataset(100, 0.5)
print(f"   -> Created {len(data_d)} records.")

# Define Fields (Note: has_missing=True is crucial now!)
fields = [
    dedupe.variables.String('name', has_missing=True),
    dedupe.variables.String('address', has_missing=True),
    dedupe.variables.String('phone', has_missing=True),
    dedupe.variables.String('email', has_missing=True),
    dedupe.variables.String('dob', has_missing=True),
    dedupe.variables.String('zip', has_missing=True)
]

if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

print("🧠 Initializing & Training...")
deduper = dedupe.Dedupe(fields)
deduper.prepare_training(data_d)
deduper.mark_pairs(training_data)
deduper.train()

with open(SETTINGS_FILE, 'wb') as f:
    deduper.write_settings(f)

print("🧩 Clustering...")
clustered_dupes = deduper.partition(data_d, threshold=0.5)

# ==========================================
# 3. View Results
# ==========================================
output = []
for cluster_id, (members, scores) in enumerate(clustered_dupes):
    for member_id, score in zip(members, scores):
        row = data_d[member_id].copy()
        row['cluster_id'] = cluster_id
        row['score'] = round(score, 2)
        row['id'] = member_id
        output.append(row)

df = pd.DataFrame(output)

# Filter for interesting duplicates (Cluster size > 1)
dupes = df[df.duplicated('cluster_id', keep=False)].sort_values('cluster_id')

print(f"\n✅ Found {len(clustered_dupes)} clusters.")
print("👀 Inspecting Complex Matches (Notice Phones & Emails):")
# Reorder columns for readability
cols = ['cluster_id', 'id', 'score', 'name', 'phone', 'email', 'dob']
print(dupes[cols].head(30).to_string(index=False))

🌪️ Generating Complex Dataset (Typos, Formats, Nulls)...
   -> Created 150 records.
🧠 Initializing & Training...
🧩 Clustering...

✅ Found 100 clusters.
👀 Inspecting Complex Matches (Notice Phones & Emails):
 cluster_id  id  score               name          phone                  email        dob
          0   1    0.5     Charles Garcia   395-358-4210      cgarcia@gmail.com 1967-03-18
          0 107    0.5      Chares Garcia     3953584210      cgarcia@gmail.com 1967-03-18
          1   2    0.5  Elizabeth Johnson   567-458-4763   ejohnson@outlook.com 1960-09-29
          1 140    0.5  Elizabeht Johnson     5674584763   ejohnson@outlookc.om 1960-09-29
          2   3    0.5        David Brown   453-349-5017     dbrown@hotmail.com 1996-11-13
          2 104    0.5           D. Brown     4533495017     dbrown@hotmail.cmo 13/11/1996
          3   5    0.5      William Smith   485-235-3269     wsmith@hotmail.com 2000-09-07
          3 123    0.5       Wiliam Smith (485) 235-3269         

In [11]:
import os
import random
import string
import dedupe
import dedupe.variables
import json
import pandas as pd
import datetime
import time

# ==========================================
# 1. Expanded Data Generator (For Scale) 🏭
# ==========================================
def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s, intensity=1):
    """Introduces random typos."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    for _ in range(intensity):
        if len(s_list) < 2: break
        idx = random.randint(0, len(s_list) - 2)
        ctype = random.choice([0, 0, 1]) # 0=Swap, 1=Delete
        if ctype == 0: s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
        elif ctype == 1: del s_list[idx]
    return "".join(s_list)

def corrupt_phone(phone):
    digits = "".join(filter(str.isdigit, phone))
    style = random.choice(['dot', 'plain', 'paren', 'space'])
    if style == 'dot': return f"{digits[:3]}.{digits[3:6]}.{digits[6:]}"
    if style == 'plain': return digits
    if style == 'space': return f"{digits[:3]} {digits[3:6]} {digits[6:]}"
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

def generate_large_dataset(target_rows=2000, duplicate_ratio=0.3):
    
    # Expanded Data Pools to avoid accidental collisions
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa", "Nancy", "Betty", "Sandra", "Margaret", "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda", "Melissa", "Deborah", "Stephanie", "Rebecca", "Laura", "Sharon", "Cynthia", "Kathleen", "Amy", "Shirley", "Angela", "Helen", "Anna", "Brenda", "Pamela", "Nicole", "Emma", "Samantha"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "icloud.com", "aol.com", "zoho.com"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct", "Washington Blvd", "Lakeview Dr", "Hillside Ave", "Park Place", "Broadway", "Highland Dr", "Sunset Blvd", "Willow Way", "River Rd", "Elm St", "Forest Ln", "Madison Ave", "Lincoln St", "Church St", "3rd St"]
    cities = ["New York", "Chicago", "Los Angeles", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville", "Fort Worth", "Columbus", "Charlotte", "San Francisco", "Indianapolis", "Seattle", "Denver", "Washington"]
    
    data_d = {}
    ground_truth = {}
    counter = 0
    
    # Calculate how many unique people vs duplicates we need
    # If ratio is 0.3, then 30% of final rows are duplicates.
    # We generate X base records, then add Y duplicates.
    # Target = Base + (Base * Ratio) -> Base = Target / (1 + Ratio)
    # Actually, simpler logic: duplicate_ratio is % of base records to duplicate.
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    
    print(f"   -> Generating {num_base_records} unique people...")

    # --- 1. Base Records ---
    for _ in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        phone = f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}"
        dob = random_date()
        street_num = random.randint(1, 9999)
        
        # Inject random suffix to email to ensure uniqueness (john.smith83@...)
        email_suffix = random.randint(1, 999)
        
        record = {
            'name': f"{fn} {ln}",
            'address': f"{street_num} {random.choice(streets)}",
            'city': random.choice(cities),
            'phone': phone,
            'email': f"{fn[0].lower()}{ln.lower()}{email_suffix}@{random.choice(domains)}",
            'dob': dob.strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]

    # --- 2. Generate Noisy Duplicates ---
    num_dupes = target_rows - num_base_records
    print(f"   -> Generating {num_dupes} duplicates with noise...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for original_id in ids_to_dupe:
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Apply Logic: Randomly corrupt fields
        if random.random() > 0.6: new_rec['name'] = corrupt_string(new_rec['name'])
        if random.random() > 0.5: new_rec['phone'] = corrupt_phone(new_rec['phone'])
        if random.random() > 0.8: new_rec['email'] = corrupt_string(new_rec['email'])
        
        # Null injection
        if random.random() > 0.9: new_rec['email'] = None
        if random.random() > 0.9: new_rec['phone'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- 3. Generate Auto-Training Pairs ---
    # We need MORE training pairs for a larger dataset to help Blocking
    match_pairs, distinct_pairs = [], []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(50): # Increased to 50
        if not multi_grps: break
        grp = random.choice(multi_grps)
        a, b = random.sample(grp, 2)
        match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(50): # Increased to 50
        a, b = random.sample(all_ids, 2)
        is_same = False
        for g in ground_truth.values():
            if a in g and b in g: is_same = True
        if not is_same:
            distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 2. Execution 🚀
# ==========================================
SETTINGS_FILE = 'large_scale.settings'
if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

print("🌪️ Generating 2000-row Complex Dataset...")
data_d, training_data = generate_large_dataset(target_rows=2000, duplicate_ratio=0.3)
print(f"✅ Created {len(data_d)} total records.")

fields = [
    dedupe.variables.String('name', has_missing=True),
    dedupe.variables.String('address', has_missing=True),
    dedupe.variables.String('phone', has_missing=True),
    dedupe.variables.String('email', has_missing=True),
    dedupe.variables.String('dob', has_missing=True),
    dedupe.variables.String('zip', has_missing=True)
]

print("\n🧠 Training Model...")
t0 = time.time()
deduper = dedupe.Dedupe(fields)
deduper.prepare_training(data_d)
deduper.mark_pairs(training_data)
deduper.train() # This takes longer with 2000 rows as it builds blocking rules
print(f"🎓 Training complete in {time.time()-t0:.2f}s")

print("🧩 Clustering 2000 records...")
t1 = time.time()
clustered_dupes = deduper.partition(data_d, threshold=0.5)
print(f"✅ Clustering complete in {time.time()-t1:.2f}s")

# ==========================================
# 3. View Results
# ==========================================
output = []
for cluster_id, (members, scores) in enumerate(clustered_dupes):
    for member_id, score in zip(members, scores):
        row = data_d[member_id].copy()
        row['cluster_id'] = cluster_id
        row['score'] = round(score, 2)
        row['id'] = member_id
        output.append(row)

df = pd.DataFrame(output)
dupes = df[df.duplicated('cluster_id', keep=False)].sort_values('cluster_id')

print(f"\n📊 Stats:")
print(f"   - Input Rows: {len(data_d)}")
print(f"   - Unique Entities Found: {len(clustered_dupes)}")
print(f"   - Duplicate Records Found: {len(dupes)}")

print("\n👀 Sample of Found Duplicates (First 15 rows):")
cols = ['cluster_id', 'id', 'score', 'name', 'phone', 'email']
print(dupes[cols].head(15).to_string(index=False))

🌪️ Generating 2000-row Complex Dataset...
   -> Generating 1538 unique people...
   -> Generating 462 duplicates with noise...
✅ Created 2000 total records.

🧠 Training Model...
🎓 Training complete in 26.73s
🧩 Clustering 2000 records...
✅ Clustering complete in 7.65s

📊 Stats:
   - Input Rows: 2000
   - Unique Entities Found: 1537
   - Duplicate Records Found: 860

👀 Sample of Found Duplicates (First 15 rows):
 cluster_id   id  score               name          phone                  email
          0    1    0.5       Emily Taylor   726-534-6858   etaylor448@yahoo.com
          0 1835    0.5       Emily Taylor   726-534-6858   etaylor448@yahooc.om
          1    2    0.5    Elizabeth Green   440-416-2292    egreen190@yahoo.com
          1 1941    0.5    Elizabeth Green   440 416 2292     egeen190@yahoo.com
          2    5    0.5    Ashley Anderson   462-872-6157 aanderson452@gmail.com
          2 1804    0.5    Ashley nAderson   462-872-6157 aanderson452@gmail.com
          3    6   

In [12]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [13]:
import os
import random
import dedupe
import dedupe.variables
import json
import pandas as pd
import datetime
import time

# ==========================================
# 1. Data Generator (2000 Rows)
# ==========================================
def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s, intensity=1):
    if not s or len(s) < 3: return s
    s_list = list(s)
    for _ in range(intensity):
        if len(s_list) < 2: break
        idx = random.randint(0, len(s_list) - 2)
        ctype = random.choice([0, 0, 1]) 
        if ctype == 0: s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
        elif ctype == 1: del s_list[idx]
    return "".join(s_list)

def corrupt_phone(phone):
    digits = "".join(filter(str.isdigit, phone))
    style = random.choice(['dot', 'plain', 'paren', 'space'])
    if style == 'dot': return f"{digits[:3]}.{digits[3:6]}.{digits[6:]}"
    if style == 'plain': return digits
    if style == 'space': return f"{digits[:3]} {digits[3:6]} {digits[6:]}"
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

def generate_large_dataset(target_rows=2000, duplicate_ratio=0.3):
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa", "Nancy", "Betty", "Sandra", "Margaret", "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda", "Melissa", "Deborah", "Stephanie", "Rebecca", "Laura", "Sharon", "Cynthia", "Kathleen", "Amy", "Shirley", "Angela", "Helen", "Anna", "Brenda", "Pamela", "Nicole", "Emma", "Samantha"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "icloud.com", "aol.com", "zoho.com"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct", "Washington Blvd", "Lakeview Dr", "Hillside Ave", "Park Place", "Broadway", "Highland Dr", "Sunset Blvd", "Willow Way", "River Rd", "Elm St", "Forest Ln", "Madison Ave", "Lincoln St", "Church St", "3rd St"]
    cities = ["New York", "Chicago", "Los Angeles", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville", "Fort Worth", "Columbus", "Charlotte", "San Francisco", "Indianapolis", "Seattle", "Denver", "Washington"]
    
    data_d = {}
    ground_truth = {}
    counter = 0
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   -> Generating {num_base_records} unique people...")

    for _ in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        phone = f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}"
        dob = random_date()
        street_num = random.randint(1, 9999)
        email_suffix = random.randint(1, 999)
        record = {
            'name': f"{fn} {ln}",
            'address': f"{street_num} {random.choice(streets)}",
            'city': random.choice(cities),
            'phone': phone,
            'email': f"{fn[0].lower()}{ln.lower()}{email_suffix}@{random.choice(domains)}",
            'dob': dob.strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]

    num_dupes = target_rows - num_base_records
    print(f"   -> Generating {num_dupes} duplicates with noise...")
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for original_id in ids_to_dupe:
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        if random.random() > 0.6: new_rec['name'] = corrupt_string(new_rec['name'])
        if random.random() > 0.5: new_rec['phone'] = corrupt_phone(new_rec['phone'])
        if random.random() > 0.8: new_rec['email'] = corrupt_string(new_rec['email'])
        if random.random() > 0.9: new_rec['email'] = None
        if random.random() > 0.9: new_rec['phone'] = None
        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    match_pairs, distinct_pairs = [], []
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(50):
        if not multi_grps: break
        grp = random.choice(multi_grps)
        a, b = random.sample(grp, 2)
        match_pairs.append((data_d[a], data_d[b]))
    all_ids = list(data_d.keys())
    for _ in range(50):
        a, b = random.sample(all_ids, 2)
        is_same = False
        for g in ground_truth.values():
            if a in g and b in g: is_same = True
        if not is_same:
            distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 2. Training and Clustering 🧠
# ==========================================
SETTINGS_FILE = 'large_scale.settings'
if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

print("🌪️ Generating 2000-row Complex Dataset...")
data_d, training_data = generate_large_dataset(target_rows=2000, duplicate_ratio=0.3)

fields = [
    dedupe.variables.String('name', has_missing=True),
    dedupe.variables.String('address', has_missing=True),
    dedupe.variables.String('phone', has_missing=True),
    dedupe.variables.String('email', has_missing=True),
    dedupe.variables.String('dob', has_missing=True),
    dedupe.variables.String('zip', has_missing=True)
]

print("\n🧠 Training Model...")
t0 = time.time()
deduper = dedupe.Dedupe(fields)
deduper.prepare_training(data_d)
deduper.mark_pairs(training_data)
deduper.train()
print(f"🎓 Training complete in {time.time()-t0:.2f}s")

print("🧩 Clustering 2000 records...")
t1 = time.time()
clustered_dupes = deduper.partition(data_d, threshold=0.5)
print(f"✅ Clustering complete in {time.time()-t1:.2f}s")

# ==========================================
# 3. Export to Excel 📊
# ==========================================
output = []
for cluster_id, (members, scores) in enumerate(clustered_dupes):
    for member_id, score in zip(members, scores):
        row = data_d[member_id].copy()
        row['cluster_id'] = cluster_id
        row['score'] = round(score, 2)
        row['id'] = member_id
        output.append(row)

# Create DataFrame
df = pd.DataFrame(output)

# Sort by cluster_id so duplicates appear next to each other
df_sorted = df.sort_values(by=['cluster_id', 'score'], ascending=[True, False])

# Reorder columns to put Cluster ID and Name first
cols = ['cluster_id', 'score', 'id', 'name', 'phone', 'email', 'address', 'city', 'zip', 'dob']
df_sorted = df_sorted[cols]

# EXCEL EXPORT
output_filename = 'dedupe_results_2000.xlsx'
print(f"\n💾 Saving results to {output_filename}...")
try:
    df_sorted.to_excel(output_filename, index=False)
    print("✅ Success! Check your folder for the Excel file.")
except Exception as e:
    print(f"❌ Error saving Excel file: {e}")
    print("   Make sure you have 'openpyxl' installed (!pip install openpyxl)")

print(f"\n📊 Quick Preview (Top 10):")
print(df_sorted[['cluster_id', 'score', 'name', 'phone']].head(10).to_string(index=False))

🌪️ Generating 2000-row Complex Dataset...
   -> Generating 1538 unique people...
   -> Generating 462 duplicates with noise...

🧠 Training Model...
🎓 Training complete in 27.87s
🧩 Clustering 2000 records...
✅ Clustering complete in 7.39s

💾 Saving results to dedupe_results_2000.xlsx...
✅ Success! Check your folder for the Excel file.

📊 Quick Preview (Top 10):
 cluster_id  score            name          phone
          0    0.5  Rebecca Flores   876-777-7674
          0    0.5   Rebecc Flores   876 777 7674
          1    0.5      Emma Scott   816-407-8078
          1    0.5      Emma Sctot   816 407 8078
          2    0.5  Shirley Wilson   384-319-2120
          2    0.5  Shirley Wilson     3843192120
          3    0.5  Cynthia Martin   555-826-8039
          3    0.5 Elizabeth White   238-667-1832
          3    0.5 Elizabeht White (238) 667-1832
          4    0.5  Kathleen Baker   552-472-4105


In [23]:
import os
import random
import dedupe
import dedupe.variables
import json
import pandas as pd
import datetime
import time
import csv

# ==========================================
# 1. Data Generator (Optimized for 200k) 🏭
# ==========================================
def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s, intensity=1):
    if not s or len(s) < 3: return s
    s_list = list(s)
    # Reducing loop complexity for speed
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx] # Swap
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx] # Delete
    return "".join(s_list)

def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    
    # Pools
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa", "Nancy", "Betty", "Sandra", "Margaret", "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda", "Melissa", "Deborah", "Stephanie", "Rebecca", "Laura", "Sharon", "Cynthia", "Kathleen", "Amy", "Shirley", "Angela", "Helen", "Anna", "Brenda", "Pamela", "Nicole", "Emma", "Samantha"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "icloud.com", "aol.com"]
    cities = ["New York", "Chicago", "Los Angeles", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville", "San Francisco", "Columbus", "Fort Worth", "Indianapolis", "Charlotte", "Seattle", "Denver", "Washington"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct", "Washington Blvd", "Lakeview Dr", "Hillside Ave", "Park Place", "Broadway"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   -> Plan: {num_base_records} unique + {target_rows - num_base_records} duplicates")

    # --- 1. Base Records ---
    print("   -> Generating unique records...")
    for i in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        street_num = random.randint(1, 9999)
        email_suffix = random.randint(1, 99999) # Higher variance for unique emails
        
        record = {
            'name': f"{fn} {ln}",
            'address': f"{street_num} {random.choice(streets)}",
            'city': random.choice(cities),
            'phone': f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}",
            'email': f"{fn[0].lower()}{ln.lower()}{email_suffix}@{random.choice(domains)}",
            'dob': random_date().strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        if i % 20000 == 0 and i > 0: print(f"      ...generated {i} base records")

    # --- 2. Generate Duplicates ---
    num_dupes = target_rows - num_base_records
    print(f"   -> Generating {num_dupes} duplicates...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Corruptions
        if random.random() > 0.6: new_rec['name'] = corrupt_string(new_rec['name'])
        if random.random() > 0.8: new_rec['email'] = None # More nulls for realism
        if random.random() > 0.8: new_rec['phone'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)
        
        if i % 20000 == 0 and i > 0: print(f"      ...generated {i} duplicates")

    # --- 3. Training Pairs (Increased for scale) ---
    print("   -> Generating training pairs...")
    match_pairs, distinct_pairs = [], []
    
    # Matches (Need more for 200k rows to help blocking)
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(200): 
        grp = random.choice(multi_grps)
        a, b = random.sample(grp, 2)
        match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(200):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 2. Execution Loop 🚀
# ==========================================
SETTINGS_FILE = 'huge_scale.settings'
if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

print("🌪️ STARTING 200,000 ROW GENERATION...")
t_gen = time.time()
data_d, training_data = generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3)
print(f"✅ Data Generation Complete in {time.time()-t_gen:.2f}s")
print(f"   Total Records: {len(data_d)}")

fields = [
    dedupe.variables.String('name', has_missing=True),
    dedupe.variables.String('address', has_missing=True),
    dedupe.variables.String('phone', has_missing=True),
    dedupe.variables.String('email', has_missing=True),
    dedupe.variables.String('dob', has_missing=True),
    dedupe.variables.String('zip', has_missing=True)
]

print("\n🧠 Training Model (This may take 2-5 minutes)...")
t0 = time.time()
deduper = dedupe.Dedupe(fields)
deduper.prepare_training(data_d)
deduper.mark_pairs(training_data)
deduper.train() 
print(f"🎓 Training complete in {time.time()-t0:.2f}s")

print("\n🧩 Clustering 200,000 records (This is the bottleneck)...")
print("   (This process is single-threaded. Please wait...)")
t1 = time.time()
clustered_dupes = deduper.partition(data_d, threshold=0.5)
print(f"✅ Clustering complete in {time.time()-t1:.2f}s")

# ==========================================
# 3. Efficient CSV Export 💾
# ==========================================
print("\n📝 Preparing Data for Export...")

# Use a list of dicts for memory efficiency
output_rows = []
for cluster_id, (members, scores) in enumerate(clustered_dupes):
    for member_id, score in zip(members, scores):
        # Only saving clustered results to save memory
        row = data_d[member_id]
        output_rows.append({
            'Cluster ID': cluster_id,
            'Confidence Score': round(score, 3),
            'ID': member_id,
            'Name': row['name'],
            'Phone': row['phone'],
            'Email': row['email'],
            'Address': row['address'],
            'City': row['city'],
            'Zip': row['zip'],
            'DOB': row['dob']
        })

# Save to CSV
output_filename = 'dedupe_results_200k.csv'
print(f"💾 Writing to {output_filename}...")

try:
    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
    print("✅ Success! Check 'dedupe_results_200k.csv'")
except Exception as e:
    print(f"❌ Error writing CSV: {e}")

# Stats
unique_entities = len(clustered_dupes)
print(f"\n📊 Final Stats:")
print(f"   - Input: 200,000 rows")
print(f"   - Unique Entities: {unique_entities}")
print(f"   - Reduction: {200000 - unique_entities} duplicates merged")

🌪️ STARTING 200,000 ROW GENERATION...
   -> Plan: 153846 unique + 46154 duplicates
   -> Generating unique records...
      ...generated 20000 base records
      ...generated 40000 base records
      ...generated 60000 base records
      ...generated 80000 base records
      ...generated 100000 base records
      ...generated 120000 base records
      ...generated 140000 base records
   -> Generating 46154 duplicates...
      ...generated 20000 duplicates
      ...generated 40000 duplicates
   -> Generating training pairs...
✅ Data Generation Complete in 6.16s
   Total Records: 200000

🧠 Training Model (This may take 2-5 minutes)...


UserWarning: The record
{'name': 'Nancy Wright', 'address': '2759 Oak Ave', 'city': 'Philadelphia', 'phone': '387-269-2972', 'email': 'nwright22889@yahoo.com', 'dob': '1992-06-29', 'zip': '93017'}
is not known to to the active learner. Make sure all `labeled_pairs` are in the data or training file of the `prepare_training()` method

In [24]:
import random
import datetime
import os
import csv
import time
import multiprocessing
import json  # Added json import
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    
    # Randomly choose corruption type: Swap adjacent chars or Delete char
    if random.random() > 0.5:
        # Swap
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        # Delete
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
        
    return "".join(s_list)

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    """
    Generates a large synthetic dataset with ground truth.
    Returns: (data_dictionary, training_pairs_dictionary)
    """
    
    # --- Data Pools (Expanded for variety) ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa", "Nancy", "Betty", "Sandra", "Margaret", "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda", "Melissa", "Deborah", "Stephanie", "Rebecca", "Laura", "Sharon", "Cynthia", "Kathleen", "Amy", "Shirley", "Angela", "Helen", "Anna", "Brenda", "Pamela", "Nicole", "Emma", "Samantha"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "icloud.com", "aol.com", "zoho.com"]
    cities = ["New York", "Chicago", "Los Angeles", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville", "San Francisco", "Columbus", "Seattle", "Denver", "Washington", "Boston", "Detroit", "Nashville", "Portland", "Las Vegas"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct", "Washington Blvd", "Lakeview Dr", "Hillside Ave", "Park Place", "Broadway", "Highland Dr", "Sunset Blvd", "Willow Way", "River Rd", "Elm St", "Forest Ln", "Madison Ave", "Lincoln St", "Church St", "3rd St"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    # Calculate counts
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        street_num = random.randint(1, 9999)
        # Add random suffix to email to prevent accidental duplicates
        email_suffix = random.randint(1, 99999)
        
        record = {
            'name': f"{fn} {ln}",
            'address': f"{street_num} {random.choice(streets)}",
            'city': random.choice(cities),
            'phone': f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}",
            'email': f"{fn[0].lower()}{ln.lower()}{email_suffix}@{random.choice(domains)}",
            'dob': random_date().strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        # Periodic printing if run in console (optional)
        if i % 50000 == 0 and i > 0: 
            print(f"      ...{i} base records created")

    # --- B. Generate Duplicates ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates...")
    
    # Pick random IDs to duplicate
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Apply corruptions (Noise)
        if random.random() > 0.6: new_rec['name'] = corrupt_string(new_rec['name'])
        if random.random() > 0.8: new_rec['email'] = None 
        if random.random() > 0.8: new_rec['phone'] = None
        
        # Corrupt address sometimes
        if random.random() > 0.9: 
             new_rec['address'] = new_rec['address'].replace("St", "Street").replace("Ave", "Avenue")

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # 1. Matches: Pick from known ground truth groups
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    
    # Generate 300 match pairs for robust training
    for _ in range(300): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # 2. Distincts: Pick random pairs and ensure they aren't actually matches
    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        
        # Check ground truth to ensure we don't accidentally pick a match
        is_match = False
        # Fast check: if they are in different ground truth buckets
        # Note: Since ground_truth keys are the original IDs, this is a simplified check
        # Ideally we check membership, but for synthetic data random selection is 99.9% distinct
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block (Fixes Windows Freeze)
# ==========================================
if __name__ == '__main__':
    # Add freeze_support for Windows executables (good practice)
    multiprocessing.freeze_support()
    
    SETTINGS_FILE = 'parallel_dedupe.settings'
    TRAINING_JSON = 'dedupe_training_data.json' # Temp file for training data
    OUTPUT_FILE = 'dedupe_results_200k.csv'
    
    # 1. Configuration
    if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

    fields = [
        dedupe.variables.String('name', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('phone', has_missing=True),
        dedupe.variables.String('email', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('zip', has_missing=True)
    ]
    
    # ⚠️ Safe Mode: Use 1 Core to verify it works (prevent freezing)
    NUM_CORES = 1  
    print(f"🛡️ Safe Mode: {NUM_CORES} Core (Prevents Windows Deadlock)")

    # 2. Generate Data
    print("🌪️ Generating Data...")
    t_gen = time.time()
    data_d, training_data = generate_huge_dataset(200000, 0.3)
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    # 2b. Save Training Data to JSON (The Fix!)
    # We must save this to a file so prepare_training can load and index it.
    # Otherwise, large datasets will ignore these records in the random sample.
    print(f"💾 Saving training pairs to {TRAINING_JSON}...")
    with open(TRAINING_JSON, 'w') as f:
        json.dump(training_data, f)

    # 3. Initialize & Train
    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    # Initialize with num_cores (Using 1 for safety now)
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    # ⚠️ Pass the TRAINING_FILE here! 
    # This forces Dedupe to learn from our specific generated pairs
    with open(TRAINING_JSON, 'r') as f:
        deduper.prepare_training(data_d, training_file=f)
    
    # NOTE: deduper.mark_pairs() is NOT needed because prepare_training loads the file
    
    print("🎓 Training...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")

    # 4. Cluster
    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    # 5. Export
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name'],
                'Phone': row['phone'],
                'Email': row['email'],
                'Address': row['address'],
                'City': row['city'],
                'Zip': row['zip'],
                'DOB': row['dob']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

🛡️ Safe Mode: 1 Core (Prevents Windows Deadlock)
🌪️ Generating Data...
   [Helper] Generating 153846 unique records...
      ...50000 base records created
      ...100000 base records created
      ...150000 base records created
   [Helper] Generating 46154 duplicates...
   [Helper] Generating training pairs...
✅ Generated in 2.33s
💾 Saving training pairs to dedupe_training_data.json...
🧠 Initializing Dedupe...


KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq/backend/cython/_zmq.py", line 179, in zmq.backend.cython._zmq._check_rc
    PyErr_CheckSignals()
^^^^^^^^^^^
KeyboardInterrupt: 


In [25]:
import random
import datetime
import os
import csv
import time
import multiprocessing
import json  # Added json import
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    
    # Randomly choose corruption type: Swap adjacent chars or Delete char
    if random.random() > 0.5:
        # Swap
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        # Delete
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
        
    return "".join(s_list)

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    """
    Generates a large synthetic dataset with ground truth.
    Returns: (data_dictionary, training_pairs_dictionary)
    """
    
    # --- Data Pools (Expanded for variety) ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa", "Nancy", "Betty", "Sandra", "Margaret", "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda", "Melissa", "Deborah", "Stephanie", "Rebecca", "Laura", "Sharon", "Cynthia", "Kathleen", "Amy", "Shirley", "Angela", "Helen", "Anna", "Brenda", "Pamela", "Nicole", "Emma", "Samantha"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "icloud.com", "aol.com", "zoho.com"]
    cities = ["New York", "Chicago", "Los Angeles", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville", "San Francisco", "Columbus", "Seattle", "Denver", "Washington", "Boston", "Detroit", "Nashville", "Portland", "Las Vegas"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct", "Washington Blvd", "Lakeview Dr", "Hillside Ave", "Park Place", "Broadway", "Highland Dr", "Sunset Blvd", "Willow Way", "River Rd", "Elm St", "Forest Ln", "Madison Ave", "Lincoln St", "Church St", "3rd St"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    # Calculate counts
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        street_num = random.randint(1, 9999)
        # Add random suffix to email to prevent accidental duplicates
        email_suffix = random.randint(1, 99999)
        
        record = {
            'name': f"{fn} {ln}",
            'address': f"{street_num} {random.choice(streets)}",
            'city': random.choice(cities),
            'phone': f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}",
            'email': f"{fn[0].lower()}{ln.lower()}{email_suffix}@{random.choice(domains)}",
            'dob': random_date().strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        # Periodic printing if run in console (optional)
        if i % 50000 == 0 and i > 0: 
            print(f"      ...{i} base records created")

    # --- B. Generate Duplicates ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates...")
    
    # Pick random IDs to duplicate
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Apply corruptions (Noise)
        if random.random() > 0.6: new_rec['name'] = corrupt_string(new_rec['name'])
        if random.random() > 0.8: new_rec['email'] = None 
        if random.random() > 0.8: new_rec['phone'] = None
        
        # Corrupt address sometimes
        if random.random() > 0.9: 
             new_rec['address'] = new_rec['address'].replace("St", "Street").replace("Ave", "Avenue")

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # 1. Matches: Pick from known ground truth groups
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    
    # Generate 300 match pairs for robust training
    for _ in range(300): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # 2. Distincts: Pick random pairs and ensure they aren't actually matches
    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        
        # Check ground truth to ensure we don't accidentally pick a match
        is_match = False
        # Fast check: if they are in different ground truth buckets
        # Note: Since ground_truth keys are the original IDs, this is a simplified check
        # Ideally we check membership, but for synthetic data random selection is 99.9% distinct
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block (Fixes Windows Freeze)
# ==========================================
if __name__ == '__main__':
    # Add freeze_support for Windows executables (good practice)
    multiprocessing.freeze_support()
    
    SETTINGS_FILE = 'parallel_dedupe.settings'
    TRAINING_JSON = 'dedupe_training_data.json' # Temp file for training data
    OUTPUT_FILE = 'dedupe_results_200k.csv'
    
    # 1. Configuration
    if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

    fields = [
        dedupe.variables.String('name', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('phone', has_missing=True),
        dedupe.variables.String('email', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('zip', has_missing=True)
    ]
    
    # ⚠️ Safe Mode: Use 1 Core to verify it works (prevent freezing)
    NUM_CORES = 1  
    print(f"🛡️ Safe Mode: {NUM_CORES} Core (Prevents Windows Deadlock)")

    # 2. Generate Data
    print("🌪️ Generating Data...")
    t_gen = time.time()
    data_d, training_data = generate_huge_dataset(200000, 0.3)
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    # 2b. Save Training Data to JSON (The Fix!)
    # We must save this to a file so prepare_training can load and index it.
    # Otherwise, large datasets will ignore these records in the random sample.
    print(f"💾 Saving training pairs to {TRAINING_JSON}...")
    with open(TRAINING_JSON, 'w') as f:
        json.dump(training_data, f)

    # 3. Initialize & Train
    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    # Initialize with num_cores (Using 1 for safety now)
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    # ⚠️ Pass the TRAINING_FILE here! 
    # This forces Dedupe to learn from our specific generated pairs
    print(f"   ...Feeding {len(data_d)} records into engine (this takes ~3-5 mins)...") # Added explicit wait message
    with open(TRAINING_JSON, 'r') as f:
        deduper.prepare_training(data_d, training_file=f)
    
    # NOTE: deduper.mark_pairs() is NOT needed because prepare_training loads the file
    
    print("🎓 Training...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")

    # 4. Cluster
    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    # 5. Export
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name'],
                'Phone': row['phone'],
                'Email': row['email'],
                'Address': row['address'],
                'City': row['city'],
                'Zip': row['zip'],
                'DOB': row['dob']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

🛡️ Safe Mode: 1 Core (Prevents Windows Deadlock)
🌪️ Generating Data...
   [Helper] Generating 153846 unique records...
      ...50000 base records created
      ...100000 base records created
      ...150000 base records created
   [Helper] Generating 46154 duplicates...
   [Helper] Generating training pairs...
✅ Generated in 2.33s
💾 Saving training pairs to dedupe_training_data.json...
🧠 Initializing Dedupe...
   ...Feeding 200000 records into engine (this takes ~3-5 mins)...


KeyboardInterrupt: 

In [27]:
import random
import datetime
import os
import csv
import time
import multiprocessing
import json  # Added json import
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    
    # Randomly choose corruption type: Swap adjacent chars or Delete char
    if random.random() > 0.5:
        # Swap
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        # Delete
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
        
    return "".join(s_list)

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    """
    Generates a large synthetic dataset with ground truth.
    Returns: (data_dictionary, training_pairs_dictionary)
    """
    
    # --- Data Pools (Expanded for variety) ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa", "Nancy", "Betty", "Sandra", "Margaret", "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda", "Melissa", "Deborah", "Stephanie", "Rebecca", "Laura", "Sharon", "Cynthia", "Kathleen", "Amy", "Shirley", "Angela", "Helen", "Anna", "Brenda", "Pamela", "Nicole", "Emma", "Samantha"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "icloud.com", "aol.com", "zoho.com"]
    cities = ["New York", "Chicago", "Los Angeles", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville", "San Francisco", "Columbus", "Seattle", "Denver", "Washington", "Boston", "Detroit", "Nashville", "Portland", "Las Vegas"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct", "Washington Blvd", "Lakeview Dr", "Hillside Ave", "Park Place", "Broadway", "Highland Dr", "Sunset Blvd", "Willow Way", "River Rd", "Elm St", "Forest Ln", "Madison Ave", "Lincoln St", "Church St", "3rd St"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    # Calculate counts
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        street_num = random.randint(1, 9999)
        # Add random suffix to email to prevent accidental duplicates
        email_suffix = random.randint(1, 99999)
        
        record = {
            'name': f"{fn} {ln}",
            'address': f"{street_num} {random.choice(streets)}",
            'city': random.choice(cities),
            'phone': f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}",
            'email': f"{fn[0].lower()}{ln.lower()}{email_suffix}@{random.choice(domains)}",
            'dob': random_date().strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        # Periodic printing if run in console (optional)
        if i % 50000 == 0 and i > 0: 
            print(f"      ...{i} base records created")

    # --- B. Generate Duplicates ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates...")
    
    # Pick random IDs to duplicate
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Apply corruptions (Noise)
        if random.random() > 0.6: new_rec['name'] = corrupt_string(new_rec['name'])
        if random.random() > 0.8: new_rec['email'] = None 
        if random.random() > 0.8: new_rec['phone'] = None
        
        # Corrupt address sometimes
        if random.random() > 0.9: 
             new_rec['address'] = new_rec['address'].replace("St", "Street").replace("Ave", "Avenue")

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # 1. Matches: Pick from known ground truth groups
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    
    # Generate 300 match pairs for robust training
    for _ in range(300): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # 2. Distincts: Pick random pairs and ensure they aren't actually matches
    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        
        # Check ground truth to ensure we don't accidentally pick a match
        is_match = False
        # Fast check: if they are in different ground truth buckets
        # Note: Since ground_truth keys are the original IDs, this is a simplified check
        # Ideally we check membership, but for synthetic data random selection is 99.9% distinct
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block (Fixes Windows Freeze)
# ==========================================
if __name__ == '__main__':
    # Add freeze_support for Windows executables (good practice)
    multiprocessing.freeze_support()
    
    SETTINGS_FILE = 'parallel_dedupe.settings'
    TRAINING_JSON = 'dedupe_training_data.json' # Temp file for training data
    OUTPUT_FILE = 'dedupe_results_40k.csv'
    
    # 1. Configuration
    if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

    fields = [
        dedupe.variables.String('name', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('phone', has_missing=True),
        dedupe.variables.String('email', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('zip', has_missing=True)
    ]
    
    # ⚠️ Safe Mode: Use 1 Core to verify it works (prevent freezing)
    NUM_CORES = 1  
    print(f"🛡️ Safe Mode: {NUM_CORES} Core (Prevents Windows Deadlock)")

    # 2. Generate Data
    print("🌪️ Generating Data...")
    t_gen = time.time()
    data_d, training_data = generate_huge_dataset(40000, 0.3)
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    # 2b. Save Training Data to JSON (The Fix!)
    # We must save this to a file so prepare_training can load and index it.
    # Otherwise, large datasets will ignore these records in the random sample.
    print(f"💾 Saving training pairs to {TRAINING_JSON}...")
    with open(TRAINING_JSON, 'w') as f:
        json.dump(training_data, f)

    # 3. Initialize & Train
    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    # Initialize with num_cores (Using 1 for safety now)
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    # ⚠️ Pass the TRAINING_FILE here! 
    # This forces Dedupe to learn from our specific generated pairs
    print(f"   ...Feeding {len(data_d)} records into engine (this takes ~1-2 mins)...") # Added explicit wait message
    with open(TRAINING_JSON, 'r') as f:
        deduper.prepare_training(data_d, training_file=f)
    
    # NOTE: deduper.mark_pairs() is NOT needed because prepare_training loads the file
    
    print("🎓 Training...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")

    # 4. Cluster
    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    # 5. Export
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name'],
                'Phone': row['phone'],
                'Email': row['email'],
                'Address': row['address'],
                'City': row['city'],
                'Zip': row['zip'],
                'DOB': row['dob']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

🛡️ Safe Mode: 1 Core (Prevents Windows Deadlock)
🌪️ Generating Data...
   [Helper] Generating 30769 unique records...
   [Helper] Generating 9231 duplicates...
   [Helper] Generating training pairs...
✅ Generated in 0.47s
💾 Saving training pairs to dedupe_training_data.json...
🧠 Initializing Dedupe...
   ...Feeding 40000 records into engine (this takes ~1-2 mins)...


KeyboardInterrupt: 

In [28]:
import random
import datetime
import os
import csv
import time
import multiprocessing
import json
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    """Generates a fake IBAN-like string."""
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    """
    Generates synthetic data matching the 'GI_AGG_DATA_CHURN' schema.
    Includes Individuals and Corporate entities.
    """
    
    # --- Data Pools ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        
        # 10% Chance of being a Corporate Customer (Company Ind = 'C')
        is_corporate = random.random() < 0.10
        
        if is_corporate:
            # Corporate Logic: No Gender, No DOB
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None # Corporate usually has no occupation description
        else:
            # Individual Logic
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            name = f"{fn} {ln}"
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)

        # Shared Fields
        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None # 20% have no bank acct

        record = {
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank
        }
        
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        if i % 50000 == 0 and i > 0: print(f"      ...{i} base records created")

    # --- B. Generate Duplicates (Replicating SQL Rules) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates (Applying SQL Scenario Rules)...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Scenario 1: Typos in Name/Address (Standard Fuzzy)
        if random.random() > 0.6: 
            new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        
        # Scenario 2: Address Changed, but Bank Acct matches (SQL Rule: Same Name, Bank Acct)
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
            # Logic: Dedupe should catch this via Name + Bank Acct
            
        # Scenario 3: Missing Data (SQL Null Handling)
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None
        if random.random() > 0.8: new_rec['gender'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(300): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    SETTINGS_FILE = 'dedupe_churn_settings.settings'
    TRAINING_JSON = 'dedupe_churn_training.json'
    OUTPUT_FILE = 'dedupe_churn_results.csv'
    
    if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

    # Updated Fields based on Churn File Schema
    fields = [
        dedupe.variables.String('name_only', has_missing=True),
        dedupe.variables.String('gender', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('occupation', has_missing=True),
        dedupe.variables.String('bank_acct_no', has_missing=True)
    ]
    
    NUM_CORES = 1 # Safe Mode
    print(f"🛡️ Safe Mode: {NUM_CORES} Core")

    print("🌪️ Generating Data (Churn Scenario)...")
    t_gen = time.time()
    data_d, training_data = generate_huge_dataset(40000, 0.3) # Using 40k for test
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    print(f"💾 Saving training pairs...")
    with open(TRAINING_JSON, 'w') as f:
        json.dump(training_data, f)

    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    print(f"   ...Feeding {len(data_d)} records...")
    with open(TRAINING_JSON, 'r') as f:
        deduper.prepare_training(data_d, training_file=f)
    
    print("🎓 Training...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")

    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name_only'],
                'Gender': row['gender'],
                'DOB': row['dob'],
                'Address': row['address'],
                'Occupation': row['occupation'],
                'Bank Acct': row['bank_acct_no']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

🛡️ Safe Mode: 1 Core
🌪️ Generating Data (Churn Scenario)...
   [Helper] Generating 30769 unique records...
   [Helper] Generating 9231 duplicates (Applying SQL Scenario Rules)...
   [Helper] Generating training pairs...
✅ Generated in 3.48s
💾 Saving training pairs...
🧠 Initializing Dedupe...
   ...Feeding 40000 records...


KeyboardInterrupt: 

In [ ]:
import random
import datetime
import os
import csv
import time
import multiprocessing
import json
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    """Generates a fake IBAN-like string."""
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    """
    Generates synthetic data matching the 'GI_AGG_DATA_CHURN' schema.
    Includes Individuals and Corporate entities.
    """
    
    # --- Data Pools ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        
        # 10% Chance of being a Corporate Customer (Company Ind = 'C')
        is_corporate = random.random() < 0.10
        
        if is_corporate:
            # Corporate Logic: No Gender, No DOB
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None # Corporate usually has no occupation description
        else:
            # Individual Logic
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            name = f"{fn} {ln}"
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)

        # Shared Fields
        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None # 20% have no bank acct

        record = {
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank
        }
        
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        if i % 50000 == 0 and i > 0: print(f"      ...{i} base records created")

    # --- B. Generate Duplicates (Replicating SQL Rules) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates (Applying SQL Scenario Rules)...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Scenario 1: Typos in Name/Address (Standard Fuzzy)
        if random.random() > 0.6: 
            new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        
        # Scenario 2: Address Changed, but Bank Acct matches (SQL Rule: Same Name, Bank Acct)
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
            # Logic: Dedupe should catch this via Name + Bank Acct
            
        # Scenario 3: Missing Data (SQL Null Handling)
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None
        if random.random() > 0.8: new_rec['gender'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(300): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    SETTINGS_FILE = 'dedupe_churn_settings.settings'
    TRAINING_JSON = 'dedupe_churn_training.json'
    OUTPUT_FILE = 'dedupe_churn_results.csv'
    
    if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

    # Updated Fields based on Churn File Schema
    fields = [
        dedupe.variables.String('name_only', has_missing=True),
        dedupe.variables.String('gender', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('occupation', has_missing=True),
        dedupe.variables.String('bank_acct_no', has_missing=True)
    ]
    
    NUM_CORES = 1 # Safe Mode
    print(f"🛡️ Safe Mode: {NUM_CORES} Core")

    print("🌪️ Generating Data (Churn Scenario)...")
    t_gen = time.time()
    data_d, training_data = generate_huge_dataset(1000, 0.3) # Using 1k for test
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    print(f"💾 Saving training pairs...")
    with open(TRAINING_JSON, 'w') as f:
        json.dump(training_data, f)

    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    print(f"   ...Feeding {len(data_d)} records...")
    with open(TRAINING_JSON, 'r') as f:
        deduper.prepare_training(data_d, training_file=f)
    
    print("🎓 Training...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")

    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name_only'],
                'Gender': row['gender'],
                'DOB': row['dob'],
                'Address': row['address'],
                'Occupation': row['occupation'],
                'Bank Acct': row['bank_acct_no']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

In [31]:
import os

# ⚠️ CRITICAL FIX FOR WINDOWS: Force libraries to run on 1 Core
# These must be set BEFORE importing numpy, dedupe, or sklearn
os.environ['LOKY_MAX_CPU_COUNT'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['JOBLIB_MULTIPROCESSING'] = '0'

import random
import datetime
import csv
import time
import multiprocessing
import json
import logging  # Import logging module
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    """Generates a fake IBAN-like string."""
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    """
    Generates synthetic data matching the 'GI_AGG_DATA_CHURN' schema.
    Includes Individuals and Corporate entities.
    """
    
    # --- Data Pools ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        
        # 10% Chance of being a Corporate Customer (Company Ind = 'C')
        is_corporate = random.random() < 0.10
        
        if is_corporate:
            # Corporate Logic: No Gender, No DOB
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None # Corporate usually has no occupation description
        else:
            # Individual Logic
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            name = f"{fn} {ln}"
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)

        # Shared Fields
        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None # 20% have no bank acct

        record = {
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank
        }
        
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        if i % 50000 == 0 and i > 0: print(f"      ...{i} base records created")

    # --- B. Generate Duplicates (Replicating SQL Rules) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates (Applying SQL Scenario Rules)...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Scenario 1: Typos in Name/Address (Standard Fuzzy)
        if random.random() > 0.6: 
            new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        
        # Scenario 2: Address Changed, but Bank Acct matches (SQL Rule: Same Name, Bank Acct)
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
            # Logic: Dedupe should catch this via Name + Bank Acct
            
        # Scenario 3: Missing Data (SQL Null Handling)
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None
        if random.random() > 0.8: new_rec['gender'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(300): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    # 📝 Enable Detailed Logging
    # This will show "INFO" level logs from Dedupe (Training progress, blocking stats)
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)
    
    SETTINGS_FILE = 'dedupe_churn_settings.settings'
    TRAINING_JSON = 'dedupe_churn_training.json'
    OUTPUT_FILE = 'dedupe_churn_results.csv'
    
    if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

    # Updated Fields based on Churn File Schema
    fields = [
        dedupe.variables.String('name_only', has_missing=True),
        dedupe.variables.String('gender', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('occupation', has_missing=True),
        dedupe.variables.String('bank_acct_no', has_missing=True)
    ]
    
    NUM_CORES = 1 # Safe Mode
    print(f"🛡️ Safe Mode: {NUM_CORES} Core")

    print("🌪️ Generating Data (Churn Scenario)...")
    t_gen = time.time()
    data_d, training_data = generate_huge_dataset(1000, 0.3) # Using 1k for test
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    print(f"💾 Saving training pairs...")
    with open(TRAINING_JSON, 'w') as f:
        json.dump(training_data, f)

    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    print(f"   ...Feeding {len(data_d)} records...")
    with open(TRAINING_JSON, 'r') as f:
        deduper.prepare_training(data_d, training_file=f)
    
    print("🎓 Training...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")

    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name_only'],
                'Gender': row['gender'],
                'DOB': row['dob'],
                'Address': row['address'],
                'Occupation': row['occupation'],
                'Bank Acct': row['bank_acct_no']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

reading training from file
reading training from file


🛡️ Safe Mode: 1 Core
🌪️ Generating Data (Churn Scenario)...
   [Helper] Generating 769 unique records...
   [Helper] Generating 231 duplicates (Applying SQL Scenario Rules)...
   [Helper] Generating training pairs...
✅ Generated in 0.01s
💾 Saving training pairs...
🧠 Initializing Dedupe...
   ...Feeding 1000 records...


Final predicate set:
Final predicate set:
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)
Final predicate set:
Final predicate set:
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)
TfidfNGramCanopyPredicate: (0.8, bank_acct_no)
TfidfNGramCanopyPredicate: (0.8, bank_acct_no)


🎓 Training...


Final predicate set:
Final predicate set:
TfidfNGramCanopyPredicate: (0.8, bank_acct_no)
TfidfNGramCanopyPredicate: (0.8, bank_acct_no)
(SimplePredicate: (twoGramFingerprint, address), SimplePredicate: (commonSixGram, name_only))
(SimplePredicate: (twoGramFingerprint, address), SimplePredicate: (commonSixGram, name_only))


✅ Trained in 32.23s
🧩 Clustering...
✅ Clustered in 0.20s
💾 Saving CSV...
🎉 Done. Saved to dedupe_churn_results.csv


In [32]:
import os

# ⚠️ CRITICAL FIX FOR WINDOWS: Force libraries to run on 1 Core
# These must be set BEFORE importing numpy, dedupe, or sklearn
os.environ['LOKY_MAX_CPU_COUNT'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['JOBLIB_MULTIPROCESSING'] = '0'

import random
import datetime
import csv
import time
import multiprocessing
import json
import logging  # Import logging module
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    """Generates a fake IBAN-like string."""
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    """
    Generates synthetic data matching the 'GI_AGG_DATA_CHURN' schema.
    Includes Individuals and Corporate entities.
    """
    
    # --- Data Pools ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        
        # 10% Chance of being a Corporate Customer (Company Ind = 'C')
        is_corporate = random.random() < 0.10
        
        if is_corporate:
            # Corporate Logic: No Gender, No DOB
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None # Corporate usually has no occupation description
        else:
            # Individual Logic
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            name = f"{fn} {ln}"
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)

        # Shared Fields
        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None # 20% have no bank acct

        record = {
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank
        }
        
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        if i % 50000 == 0 and i > 0: print(f"      ...{i} base records created")

    # --- B. Generate Duplicates (Replicating SQL Rules) ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates (Applying SQL Scenario Rules)...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Scenario 1: Typos in Name/Address (Standard Fuzzy)
        if random.random() > 0.6: 
            new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        
        # Scenario 2: Address Changed, but Bank Acct matches (SQL Rule: Same Name, Bank Acct)
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
            # Logic: Dedupe should catch this via Name + Bank Acct
            
        # Scenario 3: Missing Data (SQL Null Handling)
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None
        if random.random() > 0.8: new_rec['gender'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(300): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    # 📝 Enable Detailed Logging
    # This will show "INFO" level logs from Dedupe (Training progress, blocking stats)
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)
    
    SETTINGS_FILE = 'dedupe_churn_settings.settings'
    TRAINING_JSON = 'dedupe_churn_training.json'
    OUTPUT_FILE = 'dedupe_churn_results.csv'
    
    # NOTE: We don't delete settings file anymore so we can reuse training!
    # if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

    # Updated Fields based on Churn File Schema
    fields = [
        dedupe.variables.String('name_only', has_missing=True),
        dedupe.variables.String('gender', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('occupation', has_missing=True),
        dedupe.variables.String('bank_acct_no', has_missing=True)
    ]
    
    NUM_CORES = 1 # Safe Mode
    print(f"🛡️ Safe Mode: {NUM_CORES} Core")

    print("🌪️ Generating Data (Churn Scenario)...")
    t_gen = time.time()
    # Using 1k records for quicker manual training example
    data_d, training_data = generate_huge_dataset(1000, 0.3) 
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    # ⚠️ MANUAL MODE: We commented out the auto-save of training data
    # print(f"💾 Saving training pairs...")
    # with open(TRAINING_JSON, 'w') as f:
    #     json.dump(training_data, f)

    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    # 1. Load existing training if available
    if os.path.exists(TRAINING_JSON):
        print(f"   Reading labeled examples from {TRAINING_JSON}...")
        with open(TRAINING_JSON, 'r') as f:
            deduper.prepare_training(data_d, training_file=f)
    else:
        deduper.prepare_training(data_d)

    # 2. START MANUAL TRAINING (Console Labeling)
    print("🎓 Starting active labeling...")
    print("   [Instructions] y: Yes, n: No, u: Unsure, f: Finished")
    dedupe.console_label(deduper)

    # 3. Save the training data for next time
    print(f"💾 Saving manual training to {TRAINING_JSON}...")
    with open(TRAINING_JSON, 'w') as tf:
        deduper.write_training(tf)

    print("🎓 Training Model...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")
    
    # Save settings (blocking rules)
    with open(SETTINGS_FILE, 'wb') as sf:
        deduper.write_settings(sf)

    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name_only'],
                'Gender': row['gender'],
                'DOB': row['dob'],
                'Address': row['address'],
                'Occupation': row['occupation'],
                'Bank Acct': row['bank_acct_no']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

🛡️ Safe Mode: 1 Core
🌪️ Generating Data (Churn Scenario)...
   [Helper] Generating 769 unique records...
   [Helper] Generating 231 duplicates (Applying SQL Scenario Rules)...
   [Helper] Generating training pairs...
✅ Generated in 0.01s
🧠 Initializing Dedupe...


reading training from file
reading training from file
reading training from file


   Reading labeled examples from dedupe_churn_training.json...


Final predicate set:
Final predicate set:
Final predicate set:
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)
Final predicate set:
Final predicate set:
Final predicate set:
TfidfNGramCanopyPredicate: (0.8, bank_acct_no)
TfidfNGramCanopyPredicate: (0.8, bank_acct_no)
TfidfNGramCanopyPredicate: (0.8, bank_acct_no)
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)


🎓 Starting active labeling...
   [Instructions] y: Yes, n: No, u: Unsure, f: Finished


...

 f


Finished labeling
Final predicate set:
Final predicate set:
Final predicate set:
SimplePredicate: (doubleMetaphone, name_only)
SimplePredicate: (doubleMetaphone, name_only)
SimplePredicate: (doubleMetaphone, name_only)
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)
TfidfTextCanopyPredicate: (0.8, address)
SimplePredicate: (sameSevenCharStartPredicate, bank_acct_no)
SimplePredicate: (sameSevenCharStartPredicate, bank_acct_no)
SimplePredicate: (sameSevenCharStartPredicate, bank_acct_no)
SimplePredicate: (commonThreeTokens, name_only)
SimplePredicate: (commonThreeTokens, name_only)
SimplePredicate: (commonThreeTokens, name_only)


💾 Saving manual training to dedupe_churn_training.json...
🎓 Training Model...


Final predicate set:
Final predicate set:
Final predicate set:
SimplePredicate: (doubleMetaphone, name_only)
SimplePredicate: (doubleMetaphone, name_only)
SimplePredicate: (doubleMetaphone, name_only)
(SimplePredicate: (fingerprint, address), SimplePredicate: (tokenFieldPredicate, name_only))
(SimplePredicate: (fingerprint, address), SimplePredicate: (tokenFieldPredicate, name_only))
(SimplePredicate: (fingerprint, address), SimplePredicate: (tokenFieldPredicate, name_only))
SimplePredicate: (sameSevenCharStartPredicate, bank_acct_no)
SimplePredicate: (sameSevenCharStartPredicate, bank_acct_no)
SimplePredicate: (sameSevenCharStartPredicate, bank_acct_no)
SimplePredicate: (commonThreeTokens, name_only)
SimplePredicate: (commonThreeTokens, name_only)
SimplePredicate: (commonThreeTokens, name_only)


✅ Trained in 294.28s
🧩 Clustering...
✅ Clustered in 0.22s
💾 Saving CSV...
🎉 Done. Saved to dedupe_churn_results.csv


In [ ]:
#Parallel Not working

In [18]:
import os
import random
import dedupe
import dedupe.variables
import json
import csv
import datetime
import time
import multiprocessing # Added to detect CPU count

# ==========================================
# 0. Parallel Processing Setup ⚡
# ==========================================
# Detect available CPU cores
# We usually leave 1 core free for the OS if possible, otherwise use all
available_cores = os.cpu_count()
NUM_CORES = 4

print(f"⚡ Parallel Processing Enabled: Using {NUM_CORES} out of {available_cores} CPU cores.")

# ==========================================
# 1. Data Generator (200k Rows)
# ==========================================
def random_date(start_year=1950, end_year=2005):
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    if not s or len(s) < 3: return s
    s_list = list(s)
    # Simplified corruption for generation speed
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3):
    
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Charles", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa", "Nancy", "Betty", "Sandra", "Margaret", "Ashley", "Kimberly", "Emily", "Donna", "Michelle", "Carol", "Amanda", "Melissa", "Deborah", "Stephanie", "Rebecca", "Laura", "Sharon", "Cynthia", "Kathleen", "Amy", "Shirley", "Angela", "Helen", "Anna", "Brenda", "Pamela", "Nicole", "Emma", "Samantha"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts"]
    domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "icloud.com"]
    cities = ["New York", "Chicago", "Los Angeles", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville", "San Francisco", "Columbus", "Seattle", "Denver", "Washington"]
    streets = ["Main St", "Oak Ave", "Maple Dr", "Pine Ln", "Cedar Ct", "Washington Blvd", "Lakeview Dr", "Hillside Ave", "Park Place", "Broadway"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   -> Generating {num_base_records} unique records...")

    # Batch generation logic for speed
    for i in range(num_base_records):
        counter += 1
        fn = random.choice(firsts)
        ln = random.choice(lasts)
        street_num = random.randint(1, 9999)
        email_suffix = random.randint(1, 99999)
        
        record = {
            'name': f"{fn} {ln}",
            'address': f"{street_num} {random.choice(streets)}",
            'city': random.choice(cities),
            'phone': f"{random.randint(200,900)}-{random.randint(200,900)}-{random.randint(1000,9999)}",
            'email': f"{fn[0].lower()}{ln.lower()}{email_suffix}@{random.choice(domains)}",
            'dob': random_date().strftime("%Y-%m-%d"),
            'zip': str(random.randint(10000, 99999))
        }
        data_d[counter] = record
        ground_truth[counter] = [counter]

    num_dupes = target_rows - num_base_records
    print(f"   -> Generating {num_dupes} duplicates...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        if random.random() > 0.6: new_rec['name'] = corrupt_string(new_rec['name'])
        if random.random() > 0.8: new_rec['email'] = None 
        if random.random() > 0.8: new_rec['phone'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)

    print("   -> Generating training pairs...")
    match_pairs, distinct_pairs = [], []
    
    # Generate sufficient training data for the larger dataset
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(300): 
        grp = random.choice(multi_grps)
        a, b = random.sample(grp, 2)
        match_pairs.append((data_d[a], data_d[b]))

    all_ids = list(data_d.keys())
    for _ in range(300):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 2. Execution Loop (Multi-Core) 🚀
# ==========================================
SETTINGS_FILE = 'parallel_dedupe.settings'
if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

print(f"🌪️ STARTING 200,000 ROW GENERATION...")
t_gen = time.time()
data_d, training_data = generate_huge_dataset(target_rows=200000, duplicate_ratio=0.3)
print(f"✅ Data Generation Complete in {time.time()-t_gen:.2f}s")

fields = [
    dedupe.variables.String('name', has_missing=True),
    dedupe.variables.String('address', has_missing=True),
    dedupe.variables.String('phone', has_missing=True),
    dedupe.variables.String('email', has_missing=True),
    dedupe.variables.String('dob', has_missing=True),
    dedupe.variables.String('zip', has_missing=True)
]

print(f"\n🧠 Initializing Dedupe with {NUM_CORES} Cores...")
t0 = time.time()

# --- THE PARALLELIZATION HAPPENS HERE ---
deduper = dedupe.Dedupe(fields) 
# ----------------------------------------

deduper.prepare_training(data_d)
deduper.mark_pairs(training_data)

print("🎓 Training model (Parallelized)...")
deduper.train() 
print(f"✅ Training complete in {time.time()-t0:.2f}s")

print(f"\n🧩 Clustering 200,000 records (Parallelized)...")
print("   (This will utilize multiple CPUs. Check your Task Manager!)")
t1 = time.time()

# The partition method natively uses the num_cores defined in initialization
clustered_dupes = deduper.partition(data_d, threshold=0.5)

print(f"✅ Clustering complete in {time.time()-t1:.2f}s")

# ==========================================
# 3. Efficient CSV Export 💾
# ==========================================
print("\n📝 Preparing Data for Export...")

output_rows = []
# Generator expression for speed
for cluster_id, (members, scores) in enumerate(clustered_dupes):
    for member_id, score in zip(members, scores):
        row = data_d[member_id]
        output_rows.append({
            'Cluster ID': cluster_id,
            'Confidence Score': round(score, 3),
            'ID': member_id,
            'Name': row['name'],
            'Phone': row['phone'],
            'Email': row['email'],
            'Address': row['address'],
            'City': row['city'],
            'Zip': row['zip'],
            'DOB': row['dob']
        })

output_filename = 'dedupe_results_parallel_200k.csv'
print(f"💾 Writing {len(output_rows)} rows to {output_filename}...")

try:
    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
    print(f"✅ Success! File saved.")
except Exception as e:
    print(f"❌ Error writing CSV: {e}")

print(f"\n📊 Performance Summary:")
print(f"   - Rows Processed: 200,000")
print(f"   - Cores Used: {NUM_CORES}")
print(f"   - Total Clusters Found: {len(clustered_dupes)}")

⚡ Parallel Processing Enabled: Using 4 out of 20 CPU cores.
🌪️ STARTING 200,000 ROW GENERATION...
   -> Generating 153846 unique records...
   -> Generating 46154 duplicates...
   -> Generating training pairs...
✅ Data Generation Complete in 6.06s

🧠 Initializing Dedupe with 4 Cores...


KeyboardInterrupt: 

In [22]:
import os
import dedupe
import dedupe.variables
import csv
import time
import multiprocessing

# 1. IMPORT YOUR HELPER FILE
# This makes the functions visible to Windows sub-processes
import my_dedupe_helpers as helper 

# 2. CONFIGURATION
SETTINGS_FILE = 'parallel_dedupe.settings'
OUTPUT_FILE = 'dedupe_results_200k.csv'

if os.path.exists(SETTINGS_FILE): os.remove(SETTINGS_FILE)

fields = [
    dedupe.variables.String('name', has_missing=True),
    dedupe.variables.String('address', has_missing=True),
    dedupe.variables.String('phone', has_missing=True),
    dedupe.variables.String('email', has_missing=True),
    dedupe.variables.String('dob', has_missing=True),
    dedupe.variables.String('zip', has_missing=True)
]

# 3. THE "MAIN" GUARD (CRITICAL FOR WINDOWS)
# You must wrap the execution code in this block
if __name__ == '__main__':
    
    # Set Cores (Safe number for Windows Jupyter is usually 4-6)
    NUM_CORES = 4 
    print(f"⚡ Parallel Mode: {NUM_CORES} Cores")

    # Generate Data
    print("🌪️ Generating Data...")
    t_gen = time.time()
    # Calling the function from the external file
    data_d, training_data = helper.generate_huge_dataset(200000, 0.3)
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")

    print("🧠 Initializing Parallel Dedupe...")
    t0 = time.time()
    
    # Initialize with num_cores
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    deduper.prepare_training(data_d)
    deduper.mark_pairs(training_data)
    
    print("🎓 Training...")
    deduper.train()
    print(f"✅ Trained in {time.time()-t0:.2f}s")

    print("🧩 Clustering (Parallel)...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    # Export Code (Same as before)
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name'],
                'Address': row['address']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print("🎉 Done.")

⚡ Parallel Mode: 4 Cores
🌪️ Generating Data...
   [Helper] Generating 153846 unique records...
      ...50000 base records created
      ...100000 base records created
      ...150000 base records created
   [Helper] Generating 46154 duplicates...
   [Helper] Generating training pairs...
✅ Generated in 6.05s
🧠 Initializing Parallel Dedupe...


KeyboardInterrupt: 